In [1]:
!pip install datasets
!pip install evaluate
!pip install fsspec==2023.9.2
!pip install wordcloud
!pip install accelerate>=0.21.0



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



⚙️ **Requerimientos importantes sobre el ejercicio**

- El notebook debe ejecutarse **de principio a fin sin intervención manual**.
- Si utilizas librerías que no están incluidas por defecto en Google Colab, **asegúrate de instalarlas dentro del notebook** (por ejemplo: `!pip install ...`).

- Algunas celdas incluyen identificadores especiales que indican ciertas normas que **debes** respetar:
 - `#NO-MODIFY: DATA LOAD`  
    🔒 **No modifiques** el contenido de esta celda.

  - `#NO-MODIFY: VARIABLE NAME`  
    ✏️ Puedes modificar o añadir información **dentro de la celda**, pero **sin cambiar el nombre de la variable asignada**. No incluyas más variables de las existentes en la celda.

  - `#MODIFY: ADD INFO TO SOLVE FUNCTION`  
    🔧 Puedes modificar el **interior de la función** para resolver la tarea, pero **no cambies su nombre, la cabecera ni el `return`**.



## Imports

In [2]:
import numpy as np
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\b39141\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [3]:
# Add your imports here
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from nltk.corpus import stopwords
from collections import Counter
import re
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from datasets import load_dataset, Dataset, DatasetDict, ClassLabel
import evaluate
import torch


ValueError: Your currently installed version of Keras is Keras 3, but this is not yet supported in Transformers. Please install the backwards-compatible tf-keras package with `pip install tf-keras`.

# 🔍 Ejercicio1: Detección de profesiones en tweets

## Enunciado

En este ejercicio vamos a trabajar con un conjunto de datos procedente de medios sociales online.

Utilizaremos un subconjunto de los datos de la tarea 1 del shared task [**ProfNER**](https://temu.bsc.es/smm4h-spanish), centrada en la detección de menciones a profesiones en tweets publicados durante la pandemia del COVID-19. El objetivo original de la tarea era analizar que profesiones podrían haber sido especialmente vulnerables en el contexto de la crisis sanitaria.

Para simplificar el ejercicio, he preparado una versión reducida del dataset original. Tu tarea será entrenar un clasificador binario basado en la arquitectura Transformers, que, dado un tweet, determine si contiene una mención explícita a una profesión (etiqueta `1`) o no (etiqueta `0`).




✅ **Objetivos del ejercicio**

A lo largo de este notebook, completarás las siguientes etapas para construir un clasificador de menciones a profesiones en tweets:

1. **Análisis Exploratorio de Datos (EDA)**: Calcular estadísticas básicas del conjunto de datos (como el número de ejemplos del training set, la distribución de clases del dataset, la longitud media de los textos) o crear visualizaciones para cmprender mejor el contenido de los documentos usando wordclouds o histogramas.

2. **Selección y justificación del modelo**: Elegir un modelo del Hub de Huggingface adecuado para los datos con los que se va a trabajar y el tipo de tarea a desarrollar.

3. **Entrenamiento del clasificador**: Entrenar el modelo de forma reproducible y evaluar su rendimiento sobreel conjunto de datos de validación, incluyendo un classification score y matriz de confusion

4. **Generación de predicciones sobre el conjunte de test**: Aplicar el modelo entrenado al conjunto de test, y guardar las predicciones en un archivo `.tsv` de 2 columnas `id` y `label` separadas por tabulador

📝 **Criterios de Evaluación**

Tu trabajo será evaluado según los siguientes criterios:

| Criterio                                            | Peso  |
|-----------------------------------------------------|--------|
| 🔍 Análisis exploratorio y preprocesamiento         | 20%   |
| 🤖 Selección y justificación del modelo             | 25%   |
| 📁 Formato y validez del archivo de predicciones    | 5%    |
| ⚙️ Ejecución correcta del notebook (sin intervención) | 10%   |
| 📈 Rendimiento del modelo sobre el conjunto de test | 30%   |
| ✍️ Claridad y calidad de las explicaciones          | 10%   |



🔔 **Nota importante:**

> El rendimiento del modelo se evaluará utilizando métricas estándar como el **F1-score** sobre el conjunto de test.

> El archivo de predicciones debe respetar **estrictamente** el formato solicitado (`id` y `label`, separados por tabulador y con extensión `.tsv`).  
  ❗ Si el archivo no cumple con este formato, **el ejercicio no podrá ser evaluado en esa sección**.

> El/la estudiante con el **mayor F1-score** obtendrá la puntuación máxima en el apartado de rendimiento. El resto de calificaciones se ajustarán de forma proporcional al mejor resultado



⚙️ **Requerimientos y reglas**

- El notebook debe ejecutarse **de principio a fin sin intervención manual**.
- Si utilizas librerías que no están incluidas por defecto en Google Colab, **asegúrate de instalarlas dentro del notebook** (por ejemplo: `!pip install ...`).

- Algunas celdas incluyen identificadores especiales que indican ciertas normas que **debes** respetar:
 - `#NO-MODIFY: DATA LOAD`  
    🔒 **No modifiques** el contenido de esta celda.

  - `#NO-MODIFY: VARIABLE NAME`  
    ✏️ Puedes modificar o añadir información **dentro de la celda**, pero **sin cambiar el nombre de la variable asignada**. No incluyas más variables de las existentes en la celda.

  - `#MODIFY: ADD INFO TO SOLVE FUNCTION`  
    🔧 Puedes modificar el **interior de la función** para resolver la tarea, pero **no cambies su nombre, la cabecera ni el `return`**.


# Tu resolución (rellena las celdas marcadas)

## Obtención de datos

Descargamos los datos del [repositorio de Huggingface](https://huggingface.co/datasets/luisgasco/profner_classification_master).

In [ ]:
#NO-MODIFY: DATA LOAD
from datasets import load_dataset, Dataset, DatasetDict, ClassLabel
dataset = load_dataset("luisgasco/profner_classification_master")

El dataset contiene tres subsets:
- **train** y **validation**: Contienen el identificador del tweet, el texto, y su etiqueta, que podrá tener valor 1, si contiene una mención de una profesión; o valor 0, si no contiene una mención de una profesión.
- **test**: El test set tambiíen contiene la información de label por un requerimiento de Huggingface, pero el contenido de esta variable es siempre "-1". Es decir que deberéis predecir nuevas etiquetas una vez hayáis entrenado el modelo utilizando el train y el validation set.

## Análisis exploratorio de datos

Para hacer el análisis exploratorio de datos, transformamos cada subset a un pandas dataframe para mayor comodidad.

In [ ]:
#NO-MODIFY: DATA LOAD
dataset_train_df = dataset["train"].to_pandas()
dataset_val_df = dataset["validation"].to_pandas()
dataset_test_df = dataset["test"].to_pandas()

**Número de documentos**

Obten con la función `get_num_docs_evaluation()` el número de documentos del dataset de training y validation.

> Recuerda incorporar la información para el cálculo dentro del a siguiente celda, sin modificar los atributos de entrada ni de salida de la función, ni su nombre.

In [ ]:
#MODIFY: ADD INFO TO SOLVE FUNCTION
def get_num_docs_evaluation(dataset_df):
  # Simplemente contamos cuántas filas tiene el dataframe,
  # ya que cada fila es un documento (tweet)
  num_docs = len(dataset_df)

  # No modifiques el return
  return num_docs


Una vez generada la función, puedes utilizarla posteriormente para calcular resultados y comentarlos

In [ ]:
# Aplica la función
# Vemos cuántos documentos hay en cada subset para conocer el tamaño del dataset
print(f"Número de documentos en train: {get_num_docs_evaluation(dataset_train_df)}")
print(f"Número de documentos en validation: {get_num_docs_evaluation(dataset_val_df)}")
print(f"Número de documentos en test: {get_num_docs_evaluation(dataset_test_df)}")


**Número de documentos duplicados**

Obten con la función `detect_duplicates_evaluation()` el número de documentos duplicados del dataset de training y validation.

> Recuerda incorporar la información para el cálculo dentro del a siguiente celda, sin modificar los atributos de entrada ni de salida de la función, ni su nombre.

In [ ]:
#MODIFY: ADD INFO TO SOLVE FUNCTION
def detect_duplicates_evaluation(dataset_df):
  # Usamos .duplicated() sobre la columna 'text' para encontrar
  # tweets que tienen el mismo contenido textual repetido
  num_duplicates = dataset_df.duplicated(subset=['text']).sum()

  # No modifiques el return
  return num_duplicates


Una vez generada la función, puedes utilizarla posteriormente para calcular resultados y comentarlos

In [ ]:
# Aplica la función
# Comprobamos si hay tweets repetidos que podrían sesgar el modelo
print(f"Duplicados en train: {detect_duplicates_evaluation(dataset_train_df)}")
print(f"Duplicados en validation: {detect_duplicates_evaluation(dataset_val_df)}")


**Número de documentos por cada clase:**


Obten con la función `analyse_num_labels_evaluation()` para calcular el número de documentos de cada categoría en el dataset

> Recuerda incorporar la información para el cálculo dentro del a siguiente celda, sin modificar los atributos de entrada ni de salida de la función, ni su nombre.

In [ ]:
#MODIFY: ADD INFO TO SOLVE FUNCTION
def analyse_num_labels_evaluation(dataset_df):
  # Contamos cuántos tweets tienen etiqueta 1 (mencionan profesión)
  # y cuántos tienen etiqueta 0 (no mencionan profesión)
  num_positives = (dataset_df['label'] == 1).sum()
  num_negatives = (dataset_df['label'] == 0).sum()

  # No modifiques el return
  return num_positives, num_negatives


Una vez generada la función, puedes utilizarla posteriormente para calcular resultados y comentarlos

In [ ]:
# Aplica la función
# Analizamos si las clases están balanceadas o hay desequilibrio
pos_train, neg_train = analyse_num_labels_evaluation(dataset_train_df)
pos_val, neg_val = analyse_num_labels_evaluation(dataset_val_df)
print(f"Train - Positivos (con profesión): {pos_train}, Negativos (sin profesión): {neg_train}")
print(f"Validation - Positivos: {pos_val}, Negativos: {neg_val}")

# Gráfico de barras para ver la distribución visualmente
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
dataset_train_df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue','coral'])
axes[0].set_title('Distribución - Train')
axes[0].set_xlabel('Clase')
axes[0].set_ylabel('Frecuencia')
dataset_val_df['label'].value_counts().plot(kind='bar', ax=axes[1], color=['steelblue','coral'])
axes[1].set_title('Distribución - Validation')
axes[1].set_xlabel('Clase')
axes[1].set_ylabel('Frecuencia')
plt.tight_layout()
plt.show()


**Distribución de la longitud de los tweet en caracteres:**

In [ ]:
# Calculamos la longitud en caracteres de cada tweet
# Esto nos ayuda a decidir el max_length del tokenizer más adelante
dataset_train_df['text_length'] = dataset_train_df['text'].apply(len)
dataset_val_df['text_length'] = dataset_val_df['text'].apply(len)

# Histogramas para ver cómo se distribuyen las longitudes
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(dataset_train_df['text_length'], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_title('Longitud de tweets - Train')
axes[0].set_xlabel('Número de caracteres')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(dataset_train_df['text_length'].mean(), color='red', linestyle='--',
               label=f"Media: {dataset_train_df['text_length'].mean():.0f}")
axes[0].legend()

axes[1].hist(dataset_val_df['text_length'], bins=30, color='coral', edgecolor='black', alpha=0.7)
axes[1].set_title('Longitud de tweets - Validation')
axes[1].set_xlabel('Número de caracteres')
axes[1].set_ylabel('Frecuencia')
axes[1].axvline(dataset_val_df['text_length'].mean(), color='red', linestyle='--',
               label=f"Media: {dataset_val_df['text_length'].mean():.0f}")
axes[1].legend()
plt.tight_layout()
plt.show()

# Estadísticas resumidas de longitud
print(f"Longitud media (train): {dataset_train_df['text_length'].mean():.1f} caracteres")
print(f"Longitud media (val): {dataset_val_df['text_length'].mean():.1f} caracteres")


**Análisis de contenido de los tweets**

Para ello utiliza wordclouds

In [ ]:
# Creamos nubes de palabras para ver qué términos son más frecuentes
# en cada clase. Esto nos da intuición sobre el contenido del dataset.
stop_words = set(stopwords.words('spanish'))
# Añadimos palabras muy comunes en Twitter que no aportan información
stop_words.update(['https', 'http', 'co', 'rt', 'que', 'de', 'en', 'la', 'el',
                   'es', 'los', 'las', 'por', 'con', 'para', 'del', 'una', 'un',
                   'se', 'al', 'no', 'su', 'más', 'ya', 'lo', 'como', 'pero'])

# Juntamos todos los tweets de cada clase en un solo texto
texts_pos = ' '.join(dataset_train_df[dataset_train_df['label']==1]['text'].tolist())
texts_neg = ' '.join(dataset_train_df[dataset_train_df['label']==0]['text'].tolist())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
wc_pos = WordCloud(width=800, height=400, background_color='white',
                   stopwords=stop_words, max_words=100).generate(texts_pos)
axes[0].imshow(wc_pos, interpolation='bilinear')
axes[0].set_title('Wordcloud - Tweets CON profesión (label=1)', fontsize=14)
axes[0].axis('off')

wc_neg = WordCloud(width=800, height=400, background_color='white',
                   stopwords=stop_words, max_words=100).generate(texts_neg)
axes[1].imshow(wc_neg, interpolation='bilinear')
axes[1].set_title('Wordcloud - Tweets SIN profesión (label=0)', fontsize=14)
axes[1].axis('off')
plt.tight_layout()
plt.show()


## Tokenización

El texto del dataset no está preparado para ser introducido en un modelo Transformers. Lleva a cabo el proceso de tokenización.

In [ ]:
# IMPORTS
from transformers import AutoTokenizer


Selecciona un modelo apropiado para la tarea:

> Recuerda que en la siguiente celda sólo debes asignar un valor a model_name. No añadas más información en la celda.

### Justificación del modelo seleccionado

Se selecciona **BETO** (`dccuchile/bert-base-spanish-wwm-cased`) por las siguientes razones:

1. **Idioma**: Los tweets están en español, necesitamos un modelo preentrenado en español.
2. **Arquitectura**: BERT es ideal para clasificación de secuencias.
3. **Whole Word Masking**: BETO usa WWM, mejorando la comprensión morfológica del español.
4. **Tamaño adecuado**: 110M parámetros, viable para fine-tuning en GPU T4 de Colab.
5. **Rendimiento**: Resultados competitivos en múltiples benchmarks NLP en español.

Alternativas consideradas:
- `PlanTL-GOB-ES/roberta-base-bne`: RoBERTa español, buena alternativa.
- `bert-base-multilingual-cased`: Multilingual, pero modelos monolingües rinden mejor.


In [ ]:
#NO-MODIFY: VARIABLE NAME
model_name = 'dccuchile/bert-base-spanish-wwm-cased'


Puedes continuar con el proceso aquí:

In [ ]:
# Cargamos el tokenizer asociado al modelo que elegimos
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Definimos cómo tokenizar: padding hasta max_length y truncar si es más largo.
# Usamos 128 tokens porque los tweets son cortos (vimos en el EDA que la mayoría
# tiene menos de 280 caracteres, así que 128 tokens es suficiente)
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

# Aplicamos la tokenización a todo el dataset de una vez
tokenized_datasets = dataset.map(tokenize_function, batched=True)


In [ ]:
# Renombramos 'label' a 'labels' porque así lo espera el Trainer de HuggingFace
tokenized_datasets = tokenized_datasets.rename_column('label', 'labels')
# Convertimos a formato PyTorch para que sea compatible con el modelo
tokenized_datasets.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

# Separamos en los tres subsets
train_dataset = tokenized_datasets['train']
val_dataset = tokenized_datasets['validation']
test_dataset = tokenized_datasets['test']


## Fine-tuning

Carga el model para ser ajustado posteriormente:

In [ ]:
# Cargamos BETO con una cabeza de clasificación de 2 clases
# (0 = no profesión, 1 = profesión)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)


### Configuracion training_args

Configura los parámetros de entrenamiento del modelo.


>

> Recuerda que en la siguiente celda sólo debes asignar atributos a la variable training_args. No añadas  otras variables en la celda

In [ ]:
#NO-MODIFY: VARIABLE NAME
training_args = TrainingArguments(
    output_dir='./results',          # Carpeta donde se guardan los checkpoints
    num_train_epochs=3,              # 3 épocas suelen ser suficientes para fine-tuning
    per_device_train_batch_size=16,  # Batch de 16, funciona bien con GPU T4
    per_device_eval_batch_size=16,
    warmup_steps=100,                # Calentamiento gradual del learning rate
    weight_decay=0.01,               # Regularización para evitar overfitting
    learning_rate=2e-5,              # LR estándar para fine-tuning de BERT
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy='epoch',           # Evaluamos al final de cada época
    save_strategy='epoch',
    load_best_model_at_end=True,     # Nos quedamos con el mejor modelo
    metric_for_best_model='f1',      # Seleccionamos por F1 (métrica del ejercicio)
    seed=42,                         # Reproducibilidad
    report_to='none',                # No enviamos logs a ningún servicio externo
)


### Métricas de evaluación

Define las métricas de evaluación

In [ ]:
# Cargamos las métricas que vamos a usar para evaluar el modelo
import numpy as np
import evaluate

accuracy_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')
precision_metric = evaluate.load('precision')
recall_metric = evaluate.load('recall')

# Esta función se llama automáticamente durante el entrenamiento
# para calcular las métricas sobre el set de validación
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Tomamos la clase con mayor probabilidad como predicción
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels)
    prec = precision_metric.compute(predictions=predictions, references=labels)
    rec = recall_metric.compute(predictions=predictions, references=labels)
    return {
        'accuracy': acc['accuracy'],
        'f1': f1['f1'],
        'precision': prec['precision'],
        'recall': rec['recall']
    }


In [ ]:
# Cargamos las métricas que vamos a usar para evaluar el modelo
import numpy as np
import evaluate

accuracy_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')
precision_metric = evaluate.load('precision')
recall_metric = evaluate.load('recall')

# Esta función se llama automáticamente durante el entrenamiento
# para calcular las métricas sobre el set de validación
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Tomamos la clase con mayor probabilidad como predicción
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels)
    prec = precision_metric.compute(predictions=predictions, references=labels)
    rec = recall_metric.compute(predictions=predictions, references=labels)
    return {
        'accuracy': acc['accuracy'],
        'f1': f1['f1'],
        'precision': prec['precision'],
        'recall': rec['recall']
    }


In [ ]:
# Cargamos las métricas que vamos a usar para evaluar el modelo
import numpy as np
import evaluate

accuracy_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')
precision_metric = evaluate.load('precision')
recall_metric = evaluate.load('recall')

# Esta función se llama automáticamente durante el entrenamiento
# para calcular las métricas sobre el set de validación
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Tomamos la clase con mayor probabilidad como predicción
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels)
    prec = precision_metric.compute(predictions=predictions, references=labels)
    rec = recall_metric.compute(predictions=predictions, references=labels)
    return {
        'accuracy': acc['accuracy'],
        'f1': f1['f1'],
        'precision': prec['precision'],
        'recall': rec['recall']
    }


In [ ]:
# Cargamos las métricas que vamos a usar para evaluar el modelo
import numpy as np
import evaluate

accuracy_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')
precision_metric = evaluate.load('precision')
recall_metric = evaluate.load('recall')

# Esta función se llama automáticamente durante el entrenamiento
# para calcular las métricas sobre el set de validación
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Tomamos la clase con mayor probabilidad como predicción
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels)
    prec = precision_metric.compute(predictions=predictions, references=labels)
    rec = recall_metric.compute(predictions=predictions, references=labels)
    return {
        'accuracy': acc['accuracy'],
        'f1': f1['f1'],
        'precision': prec['precision'],
        'recall': rec['recall']
    }


### Ajuste del modelo

Lleva a cabo el ajuste del modelo:

In [ ]:
# Configuramos el Trainer, que se encarga de todo el loop de entrenamiento
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# Lanzamos el entrenamiento (esto tarda unos minutos con GPU)
trainer.train()


## Evaluacion

Una vez llevada a cabo el entrenamiento, realiza la evaluación del modelo.

In [ ]:
# Evaluamos el modelo entrenado sobre el set de validación
eval_results = trainer.evaluate()
print("Resultados de evaluación:")
for key, value in eval_results.items():
    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")

# Generamos predicciones sobre validación para ver el detalle por clase
val_predictions = trainer.predict(val_dataset)
val_preds = np.argmax(val_predictions.predictions, axis=-1)
val_labels = val_predictions.label_ids

# Classification report: precision, recall y F1 por cada clase
print("\n=== Classification Report ===")
print(classification_report(val_labels, val_preds, target_names=['No profesión (0)', 'Profesión (1)']))

# Matriz de confusión para ver errores del modelo
cm = confusion_matrix(val_labels, val_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No profesión', 'Profesión'])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap='Blues')
plt.title('Matriz de Confusión - Validación')
plt.tight_layout()
plt.show()


## Genera predicciones

Genera predicciones sobre el test set. Recuerda que el archivo que generes y adjuntes al ejercicio debe tener dos columnas:


| id         | label |
|------------|-------|
| 1234567890 | 1     |
| 1234567891 | 0     |
| 1234567892 | 0     |
| 1234567893 | 1     |

- El archivo debe estar en formato **TSV** (separado por tabuladores).
- Debe contener exactamente **dos columnas**: `id` y `label`.
- Es obligatorio incluir la **cabecera**.


In [ ]:
# Generamos predicciones sobre el test set (que no tiene etiquetas reales)
test_predictions = trainer.predict(test_dataset)
test_preds = np.argmax(test_predictions.predictions, axis=-1)

# Creamos el archivo TSV con el formato requerido: id y label
predictions_df = pd.DataFrame({
    'id': dataset_test_df['id'],
    'label': test_preds
})

# Guardamos el archivo (RECUERDA cambiar el nombre con tus apellidos)
output_filename = 'APELLIDO1_APELLIDO2_NOMBRE_ejercicio1_predicciones.tsv'
predictions_df.to_csv(output_filename, sep='\t', index=False)

# Verificamos que se guardó correctamente
print(f"Archivo guardado: {output_filename}")
print(f"Total predicciones: {len(predictions_df)}")
print(f"\nDistribución de predicciones:")
print(predictions_df['label'].value_counts())
print(f"\nPrimeras filas:")
print(predictions_df.head(10))


### Prueba de otros modelos